In [ ]:
# Import de
import numpy as np
import pandas as pd
import scipy.stats as stats
from pathlib import Path
import json

from support_functions import (
    conditional_sgm,
    convert_to_single_synteyes,
    create_retina_curvature,
    create_mgmm_data,
    nearest_psd,
    zernike_index
    )


## Model parameters

Parameters of the MGM and SGM creating the SyntEyes with retina radii.

In [12]:
# Original SyntEyes parameters
base_path = Path(__file__).parent if "__file__" in globals() else Path.cwd()
modeldata_path = base_path / "modeldata.json"
modeldata = json.loads(modeldata_path.read_text(encoding="utf-8"))

conv_ec_orig = np.array(modeldata["conv_ec_orig"])
avg_ec_orig = np.array(modeldata["avg_ec_orig"])
lens_za_orig = np.array(modeldata["lens_za_orig"])
weights_orig = np.array(modeldata["weights_orig"])
mu_orig = np.array(modeldata["mu_orig"])
cov_orig0 = np.array(modeldata["cov_orig0"])
cov_orig1 = np.array(modeldata["cov_orig1"])

cov_orig = np.zeros((2, np.shape(cov_orig0)[0], np.shape(cov_orig0)[1]))
cov_orig[0] = nearest_psd(cov_orig0)
cov_orig[1] = cov_orig1

# Retina model parameters
MU_AL_RADII = np.array([23.85, 11.89, 11.66, 10.55])
COV_AL_RADII = np.array(
    [
        [1.82, 0.56, 0.42, 0.81],
        [0.56, 0.36, 0.19, 0.18],
        [0.42, 0.19, 0.24, 0.16],
        [0.81, 0.18, 0.16, 0.48],
    ]
)

## Create n SyntEyes

In [ ]:
# Create n 3D SynthEyes

# User input to set the amount of generated eyes
n = int(input("Number of SyntEyes: "))
print("There will be " + str(n) + " SyntEyes generated")

# Create the eigencornea data
eigen_data = create_mgmm_data(mu_orig[0],mu_orig[1],cov_orig0,cov_orig1,weights_orig[0],weights_orig[1],n)

# Use the eigencornea to create the SyntEyes without retina curvature information
synteyes_orig = pd.DataFrame([])
for i in range(len(eigen_data)):
    synteyes_orig_single = convert_to_single_synteyes(
        eigen_data[i], conv_ec_orig, avg_ec_orig, lens_za_orig
    )
    synteyes_orig = pd.concat([synteyes_orig, synteyes_orig_single],ignore_index=True)

# Add the retina curvature information
synteyes_data = create_retina_curvature(synteyes_orig, MU_AL_RADII, COV_AL_RADII)

print(synteyes_data)

There will be 3 SyntEyes generated
        CCT       ACD        LT  AxialLength         VD   RT        Rla  \
0  0.553093  2.639111  4.146376    23.738706  16.200126  0.2  10.517556   
1  0.552787  2.617476  3.977226    22.587042  15.239553  0.2  10.663915   
2  0.580636  2.268051  4.284307    22.911427  15.578433  0.2   9.561701   

        Rlp     Qla  Qlp  ...  CorPostZ(8.0,-4.0)  CorPostZ(8.0,-2.0)  \
0 -6.928734 -3.1316   -1  ...           -0.000006            0.000038   
1 -7.019592 -3.1316   -1  ...            0.000002            0.000006   
2 -6.328752 -3.1316   -1  ...            0.000009            0.000002   

   CorPostZ(8.0,0.0)  CorPostZ(8.0,2.0)  CorPostZ(8.0,4.0)  CorPostZ(8.0,6.0)  \
0           0.000069           0.000032           0.000028       1.690596e-05   
1           0.000045           0.000053           0.000015      -6.940672e-07   
2           0.000021           0.000030           0.000017       8.468882e-06   

   CorPostZ(8.0,8.0)     ell_rx     ell_ry    

In [14]:
# Main Data to create a retina curvature corresponding to a provided Axial length
al = float(input("Which axial length (mm) does the eye have: "))
print(al)

# Calculate the conditional model belonging to the Axial Length
conditional_mean_sgm, conditional_cov_sgm = conditional_sgm(MU_AL_RADII, COV_AL_RADII, [0], al)
# Create a single retina curvature
rx, ry, rz = stats.multivariate_normal.rvs(
    mean=conditional_mean_sgm, cov=conditional_cov_sgm, size=1
)
print(rx, ry, rz)

24.0
12.2248649517897 11.690376488541446 10.570641260103452
